# Week 11 — EEGNet Reproducibility and Controlled Refinement

## Goal

Week 10 produced a best validation balanced accuracy of 0.615 with EEGNet,
substantially above the CNN baseline.

However, that result came from a single neural-network training run.

The first goal of Week 11 is therefore to determine whether the current EEGNet
configuration gives similar results across multiple random seeds.

### Phase 1 — Reproducibility

Run the exact Week 10 configuration across five seeds while keeping fixed:

- preprocessing
- train/validation subject split
- architecture
- optimizer
- learning rate
- dropout
- number of epochs
- evaluation metric

Only the random seed changes.

Primary outcome:

**Best validation balanced accuracy across 30 epochs**

We will report:

- mean
- standard deviation
- minimum
- maximum
- best epoch for each seed

The held-out test subjects remain untouched.

### Phase 2 — Controlled refinement

Only after Phase 1 is complete will a small number of justified training
hyperparameters be compared.

### Imports

In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import balanced_accuracy_score

### Paths and configurations

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
DROPOUT = 0.5
N_EPOCHS = 30

SEEDS = [42, 123, 456, 789, 2026]

print("Device:", DEVICE)
print("Seeds:", SEEDS)

Device: mps
Seeds: [42, 123, 456, 789, 2026]


### Load arrays and splits

In [3]:
X = np.load(
    DATA_DIR / "X_motor_imagery.npy",
    mmap_mode="r"
)

y = np.load(
    DATA_DIR / "y_motor_imagery.npy",
    mmap_mode="r"
)

subjects = np.load(
    DATA_DIR / "subjects.npy",
    mmap_mode="r"
)

split_df = pd.read_csv(
    RESULTS_DIR / "week9_subject_split.csv"
)

### Load train and validation subjects

In [4]:
train_subjects = split_df.loc[
    split_df["split"] == "train",
    "subject"
].to_numpy()

val_subjects = split_df.loc[
    split_df["split"] == "validation",
    "subject"
].to_numpy()

train_indices = np.where(
    np.isin(subjects, train_subjects)
)[0]

val_indices = np.where(
    np.isin(subjects, val_subjects)
)[0]

### Check all data loaded

In [5]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print()
print("Training subjects:", len(train_subjects))
print("Training trials:", len(train_indices))

print("Validation subjects:", len(val_subjects))
print("Validation trials:", len(val_indices))

X shape: (4898, 64, 481)
y shape: (4898,)

Training subjects: 76
Training trials: 3423
Validation subjects: 16
Validation trials: 718


### Create dataset class

In [6]:
class EEGNetDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        trial_index = self.indices[index]

        x = torch.tensor(
            self.X[trial_index],
            dtype=torch.float32
        ) * 1e6

        x = x.unsqueeze(0)

        y = torch.tensor(
            self.y[trial_index],
            dtype=torch.long
        )

        return x, y

### Create datasets

In [7]:
train_dataset = EEGNetDataset(
    X,
    y,
    train_indices
)

val_dataset = EEGNetDataset(
    X,
    y,
    val_indices
)

print("Training trials:", len(train_dataset))
print("Validation trials:", len(val_dataset))

Training trials: 3423
Validation trials: 718


### Create EEGNet class

In [8]:
class EEGNet(nn.Module):
    def __init__(
        self,
        n_channels=64,
        n_classes=2,
        F1=8,
        D=2,
        F2=16,
        temporal_kernel=40,
        separable_kernel=16,
        dropout_p=0.5
    ):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(
                1,
                F1,
                (1, temporal_kernel),
                padding="same",
                bias=False
            ),
            nn.BatchNorm2d(F1),

            nn.Conv2d(
                F1,
                F1 * D,
                (n_channels, 1),
                groups=F1,
                bias=False
            ),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),

            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout_p)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(
                F1 * D,
                F1 * D,
                (1, separable_kernel),
                padding="same",
                groups=F1 * D,
                bias=False
            ),

            nn.Conv2d(
                F1 * D,
                F2,
                (1, 1),
                bias=False
            ),
            nn.BatchNorm2d(F2),
            nn.ELU(),

            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout_p)
        )

        self.classifier = nn.Linear(
            F2 * 15,
            n_classes
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = torch.flatten(x, start_dim=1)

        return self.classifier(x)

### Train one epoch function

In [9]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        predictions = logits.argmax(dim=1)

        all_targets.extend(y_batch.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())

    mean_loss = total_loss / len(loader.dataset)

    balanced_accuracy = balanced_accuracy_score(
        all_targets,
        all_predictions
    )

    return mean_loss, balanced_accuracy

### Evaluation function

In [10]:
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item() * X_batch.size(0)

            predictions = logits.argmax(dim=1)

            all_targets.extend(y_batch.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    mean_loss = total_loss / len(loader.dataset)

    balanced_accuracy = balanced_accuracy_score(
        all_targets,
        all_predictions
    )

    return mean_loss, balanced_accuracy

### Complete seeded run function

In [11]:
def train_eegnet_run(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    train_generator = torch.Generator()
    train_generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=train_generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    model = EEGNet(
        dropout_p=DROPOUT
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    history = []

    best_val_ba = -np.inf
    best_epoch = None

    for epoch in range(1, N_EPOCHS + 1):
        train_loss, train_ba = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            DEVICE
        )

        val_loss, val_ba = evaluate(
            model,
            val_loader,
            criterion,
            DEVICE
        )

        history.append({
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_ba": train_ba,
            "val_loss": val_loss,
            "val_ba": val_ba
        })

        if val_ba > best_val_ba:
            best_val_ba = val_ba
            best_epoch = epoch

    history_df = pd.DataFrame(history)

    summary = {
        "seed": seed,
        "best_val_ba": best_val_ba,
        "best_epoch": best_epoch
    }

    return summary, history_df

### Run seed 42

In [12]:
summary_42, history_42 = train_eegnet_run(42)

print(summary_42)

/Users/heihei/Desktop/neurosignallab/.venv/lib/python3.13/site-packages/torch/nn/modules/conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1105.)
  return F.conv2d(


{'seed': 42, 'best_val_ba': 0.626723174366617, 'best_epoch': 23}


### Run rest of the seeds

In [13]:
summaries = [summary_42]
histories = [history_42]

for seed in SEEDS:
    if seed == 42:
        continue

    summary, history = train_eegnet_run(seed)

    summaries.append(summary)
    histories.append(history)

    print(
        f"Seed {seed} | "
        f"Best val BA: {summary['best_val_ba']:.3f} | "
        f"Best epoch: {summary['best_epoch']}"
    )

Seed 123 | Best val BA: 0.638 | Best epoch: 17
Seed 456 | Best val BA: 0.624 | Best epoch: 30
Seed 789 | Best val BA: 0.632 | Best epoch: 30
Seed 2026 | Best val BA: 0.620 | Best epoch: 30


### Summarize

In [14]:
summary_df = pd.DataFrame(summaries)

history_df = pd.concat(
    histories,
    ignore_index=True
)

summary_df

,seed,best_val_ba,best_epoch
0,42,0.626723,23
1,123,0.637978,17
2,456,0.624045,30
3,789,0.632289,30
4,2026,0.619939,30


### Calculate metrics

In [15]:
mean_ba = summary_df["best_val_ba"].mean()
std_ba = summary_df["best_val_ba"].std()
min_ba = summary_df["best_val_ba"].min()
max_ba = summary_df["best_val_ba"].max()

print(f"Mean best validation BA: {mean_ba:.3f}")
print(f"Standard deviation:      {std_ba:.3f}")
print(f"Minimum:                 {min_ba:.3f}")
print(f"Maximum:                 {max_ba:.3f}")

Mean best validation BA: 0.628
Standard deviation:      0.007
Minimum:                 0.620
Maximum:                 0.638


### Save as csvs

In [16]:
summary_df.to_csv(
    RESULTS_DIR / "week11_seed_results.csv",
    index=False
)

history_df.to_csv(
    RESULTS_DIR / "week11_seed_history.csv",
    index=False
)

## Phase 1 result

The Week 10 EEGNet result was reproducible across random initializations.

Five independent runs produced best validation balanced accuracies between
0.620 and 0.638:

- Seed 42: 0.627
- Seed 123: 0.638
- Seed 456: 0.624
- Seed 789: 0.632
- Seed 2026: 0.620

Mean best validation balanced accuracy:

**0.628 ± 0.007 SD**

The small standard deviation indicates that performance is relatively insensitive
to the tested random initializations.

Three of the five runs reached their highest validation balanced accuracy at
epoch 30, suggesting that the fixed 30-epoch training horizon may truncate
learning for some runs. This observation will be retained for later analysis
rather than changing the training duration immediately.

The result supports the conclusion that the improved EEGNet validation
performance observed in Week 10 was not explained by one unusually favorable
random initialization.

The held-out test subjects remain untouched.

## Phase 2 — Controlled Refinement

The goal of Phase 2 is to test a small number of predefined training changes without changing the EEGNet architecture or data split.

We will compare four configurations:

- **A — Baseline:** learning rate `1e-3`, dropout `0.50`, weight decay `0`
- **B — Lower learning rate:** learning rate `3e-4`, dropout `0.50`, weight decay `0`
- **C — Lower dropout:** learning rate `1e-3`, dropout `0.25`, weight decay `0`
- **D — Weight decay:** learning rate `1e-3`, dropout `0.50`, weight decay `1e-4`

Configuration A is the reproducibility baseline from Phase 1.

Each alternative changes only one training hyperparameter at a time.

All other factors remain fixed:
- preprocessing
- train/validation subject split
- EEGNet architecture
- batch size
- optimizer
- number of epochs
- balanced accuracy as the primary metric

Each configuration will be evaluated across the same five random seeds.

The main comparison will use the mean and standard deviation of the best validation balanced accuracy across seeds.

The held-out test subjects remain untouched.

### Make train_eeg_run() configurable

In [17]:
def train_eegnet_run(
    seed,
    learning_rate=1e-3,
    dropout=0.5,
    weight_decay=0.0
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    train_generator = torch.Generator()
    train_generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=train_generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    model = EEGNet(
        dropout_p=dropout
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    history = []

    best_val_ba = -np.inf
    best_epoch = None

    for epoch in range(1, N_EPOCHS + 1):

        train_loss, train_ba = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            DEVICE
        )

        val_loss, val_ba = evaluate(
            model,
            val_loader,
            criterion,
            DEVICE
        )

        history.append({
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_ba": train_ba,
            "val_loss": val_loss,
            "val_ba": val_ba
        })

        if val_ba > best_val_ba:
            best_val_ba = val_ba
            best_epoch = epoch

    return {
        "seed": seed,
        "best_val_ba": best_val_ba,
        "best_epoch": best_epoch
    }, pd.DataFrame(history)

### Test one seed on configuration B

In [18]:
summary_B42, history_B42 = train_eegnet_run(
    seed=42,
    learning_rate=3e-4,
    dropout=0.5,
    weight_decay=0.0
)

print(summary_B42)

{'seed': 42, 'best_val_ba': 0.6208240188772975, 'best_epoch': 27}


### Put configurations inside dataframe

In [19]:
comparison_summaries = []
comparison_histories = []

# Configuration A — already completed in Phase 1
for _, row in summary_df.iterrows():
    comparison_summaries.append({
        "config": "A",
        "seed": int(row["seed"]),
        "best_val_ba": row["best_val_ba"],
        "best_epoch": int(row["best_epoch"])
    })

history_A = history_df.copy()
history_A["config"] = "A"
comparison_histories.append(history_A)

# Configuration B — seed 42 already completed
comparison_summaries.append({
    "config": "B",
    "seed": 42,
    "best_val_ba": summary_B42["best_val_ba"],
    "best_epoch": summary_B42["best_epoch"]
})

history_B42 = history_B42.copy()
history_B42["config"] = "B"
comparison_histories.append(history_B42)

### Run remaining configs and seeds

In [20]:
configs = {
    "B": {
        "learning_rate": 3e-4,
        "dropout": 0.5,
        "weight_decay": 0.0
    },
    "C": {
        "learning_rate": 1e-3,
        "dropout": 0.25,
        "weight_decay": 0.0
    },
    "D": {
        "learning_rate": 1e-3,
        "dropout": 0.5,
        "weight_decay": 1e-4
    }
}

for config_name, params in configs.items():

    seeds_to_run = SEEDS

    if config_name == "B":
        seeds_to_run = [seed for seed in SEEDS if seed != 42]

    for seed in seeds_to_run:

        summary, history = train_eegnet_run(
            seed=seed,
            learning_rate=params["learning_rate"],
            dropout=params["dropout"],
            weight_decay=params["weight_decay"]
        )

        summary["config"] = config_name
        history["config"] = config_name

        comparison_summaries.append(summary)
        comparison_histories.append(history)

        print(
            f"{config_name} | "
            f"Seed {seed} | "
            f"Best BA: {summary['best_val_ba']:.3f} | "
            f"Epoch: {summary['best_epoch']}"
        )

B | Seed 123 | Best BA: 0.605 | Epoch: 30
B | Seed 456 | Best BA: 0.593 | Epoch: 24
B | Seed 789 | Best BA: 0.613 | Epoch: 29
B | Seed 2026 | Best BA: 0.598 | Epoch: 28
C | Seed 42 | Best BA: 0.621 | Epoch: 23
C | Seed 123 | Best BA: 0.604 | Epoch: 21
C | Seed 456 | Best BA: 0.616 | Epoch: 27
C | Seed 789 | Best BA: 0.623 | Epoch: 17
C | Seed 2026 | Best BA: 0.634 | Epoch: 18
D | Seed 42 | Best BA: 0.628 | Epoch: 19
D | Seed 123 | Best BA: 0.628 | Epoch: 17
D | Seed 456 | Best BA: 0.628 | Epoch: 28
D | Seed 789 | Best BA: 0.632 | Epoch: 30
D | Seed 2026 | Best BA: 0.627 | Epoch: 22


### Combine into dataframe

In [21]:
comparison_df = pd.DataFrame(comparison_summaries)

comparison_history_df = pd.concat(
    comparison_histories,
    ignore_index=True
)

comparison_df.sort_values(
    ["config", "seed"]
)

,config,seed,best_val_ba,best_epoch
0,A,42,0.626723,23
1,A,123,0.637978,17
2,A,456,0.624045,30
3,A,789,0.632289,30
4,A,2026,0.619939,30
5,B,42,0.620824,27
6,B,123,0.605176,30
7,B,456,0.593222,24
8,B,789,0.613334,29
9,B,2026,0.597771,28


### Compare all configurations

In [22]:
config_summary = (
    comparison_df
    .groupby("config")["best_val_ba"]
    .agg(["mean", "std", "min", "max"])
)

config_summary

,mean,std,min,max
config,,,,
A,0.628195,0.007071,0.619939,0.637978
B,0.606065,0.011238,0.593222,0.620824
C,0.619854,0.010763,0.604306,0.633825
D,0.628446,0.001923,0.626715,0.631745


## Phase 2 — Results

The controlled refinement experiments produced the following mean best validation
balanced accuracy across five seeds:

- **A — Baseline:** 0.628 ± 0.007
- **B — Lower learning rate:** 0.606 ± 0.011
- **C — Lower dropout:** 0.620 ± 0.011
- **D — Weight decay:** 0.628 ± 0.002

Lowering the learning rate reduced both performance and stability.

Reducing dropout also decreased mean performance and increased variability,
although individual runs could still perform well.

Adding weight decay (`1e-4`) did not meaningfully change mean validation
balanced accuracy compared with the baseline. However, variability across seeds
decreased substantially, from SD = 0.0071 to SD = 0.0019.

Configuration D therefore did not provide evidence of higher average accuracy,
but it produced substantially more consistent results across the tested random
initializations.

Configuration D will be retained as the current EEGNet training configuration:

- learning rate: `1e-3`
- dropout: `0.50`
- weight decay: `1e-4`
- 30 training epochs

The architecture, preprocessing, subject split, and primary evaluation metric
remain unchanged.

The held-out test subjects remain untouched.

### Save phase 2 results

In [23]:
comparison_df.to_csv(
    RESULTS_DIR / "week11_config_results.csv",
    index=False
)

comparison_history_df.to_csv(
    RESULTS_DIR / "week11_config_history.csv",
    index=False
)

## Phase 3 — Next Experiment

Phase 1 showed that the current EEGNet result is reproducible across random seeds.

Phase 2 tested three controlled training modifications. Mild weight decay preserved
mean performance while substantially reducing variability, so Configuration D was
retained as the current EEGNet training configuration.

However, the current implementation is an EEGNet-style model rather than a fully
faithful reproduction of the original EEGNet training constraints.

A separate notebook will therefore perform one final bounded model-development
stage before the architecture is frozen:

**`11b_eegnet_faithful_and_subject_analysis.ipynb`**

Phase 3 will:

- implement the missing EEGNet-specific constraints while keeping the remaining
  experimental protocol fixed
- evaluate the faithful implementation across the same five random seeds
- compare it with the current Configuration D
- examine validation performance at the individual-subject level
- determine whether the current ~0.63 validation balanced accuracy reflects a
  uniform limitation or large differences between subjects

No additional broad hyperparameter search will be performed.

The held-out test subjects remain untouched.